# Llama vs Logic Evaluation (ProofWriter, K Epochs)

This notebook compares:
1. Baseline default Llama classifier (same head output shape as logic model)
2. Logic-augmented model from this repo (V4 token-window logic path)

It is designed for Colab using a real Llama checkpoint and runs for exactly `K_EPOCHS`.

Notes:
- Both models are trained unfrozen by default.
- This is evaluation-oriented training (not unit tests).
- Dataset is ProofWriter.
- You can lower sample caps for faster experiments.
- For fair comparison in this V4 notebook, use baseline with parameter-matched no-gate control (`use_param_matched_baseline=True`).

## Set Up

In [ ]:
# Colab/runtime setup (aligned with old eval notebook repo loading)
import os
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Mirror old notebook behavior: in Colab, use secrets-driven auth.
IN_COLAB = 'google.colab' in sys.modules
userdata = None
if IN_COLAB:
    try:
        from google.colab import userdata as _colab_userdata
        userdata = _colab_userdata
    except Exception as exc:
        raise RuntimeError('Colab secrets API is unavailable; cannot continue.') from exc

    REPO_URL = 'github.com/Ancorro/AI535project.git'
    REPO_DIR = '/content/AI535project'
    TARGET_BRANCH = 'V5'

    # GitHub secret from Colab secrets only.
    try:
        gh_token = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    except Exception:
        gh_token = None
    if not gh_token:
        raise RuntimeError('Missing Colab secret GITHUB_TOKEN (or GH_TOKEN); refusing to continue.')

    auth_url = f'https://{gh_token}@{REPO_URL}'
    if os.path.isdir(REPO_DIR):
        print(f'Repo exists, updating branch {TARGET_BRANCH} ...')
        os.system(f'git -C {REPO_DIR} fetch origin {TARGET_BRANCH}')
        os.system(f'git -C {REPO_DIR} checkout {TARGET_BRANCH}')
        os.system(f'git -C {REPO_DIR} pull origin {TARGET_BRANCH}')
    else:
        print(f'Repo not found, cloning branch {TARGET_BRANCH} ...')
        os.system(f'git clone --branch {TARGET_BRANCH} --single-branch {auth_url} {REPO_DIR}')
    del auth_url

    # Hugging Face secret from Colab secrets only.
    try:
        hf_token = userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN')
    except Exception:
        hf_token = None
    if not hf_token:
        raise RuntimeError('Missing Colab secret HF_TOKEN (or HUGGINGFACE_TOKEN); refusing to continue.')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_TOKEN'] = hf_token

# Resolve repository root robustly (works for Colab and local VS Code).
cwd = Path.cwd().resolve()
repo_candidate = None
for p in [cwd, *cwd.parents]:
    if (p / 'train' / 'train.py').exists() and (p / 'logic').exists():
        repo_candidate = p
        break
if repo_candidate is None and IN_COLAB and Path('/content/AI535project').exists():
    repo_candidate = Path('/content/AI535project').resolve()
if repo_candidate is None:
    raise FileNotFoundError('Could not locate repo root containing train/train.py and logic/.')

REPO_ROOT = repo_candidate
PROJECT_ROOT = REPO_ROOT  # backwards-compatible name used in this notebook
print('REPO_ROOT:', REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Enforce W&B setup from Colab secret. Do not continue on failure.
try:
    import wandb
except Exception as exc:
    raise RuntimeError('wandb is required but not installed; install it before continuing.') from exc

if IN_COLAB:
    try:
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None
    if not wandb_key:
        raise RuntimeError('Missing Colab secret WANDB_API_KEY; refusing to continue.')
    try:
        wandb.login(key=wandb_key, relogin=True)
    except Exception as exc:
        raise RuntimeError('W&B login failed with WANDB_API_KEY secret; refusing to continue.') from exc
else:
    # Local fallback for non-Colab runs.
    wandb_key = os.getenv('WANDB_API_KEY')
    if not wandb_key:
        raise RuntimeError('Missing WANDB_API_KEY env var for local run; refusing to continue.')
    try:
        wandb.login(key=wandb_key, relogin=True)
    except Exception as exc:
        raise RuntimeError('W&B login failed with local WANDB_API_KEY; refusing to continue.') from exc

print('W&B authenticated successfully.')
if IN_COLAB:
    print('GitHub and Hugging Face tokens loaded from Colab secrets.')

# Optional installs for fresh Colab runtimes:
# !pip install -q transformers datasets accelerate wandb

## Parameters

In [ ]:
# Experiment config
from dataclasses import dataclass

@dataclass
class ExpConfig:
    model_name: str = 'meta-llama/Llama-3.1-8B'
    run_model: str = 'logic_llama'  # choose one: 'baseline_llama' or 'logic_llama'
    run_seeds: tuple[int, ...] = (42,)  # run these seeds back-to-back
    proofwriter_config: str = 'default'
    proofwriter_depth: str = 'all'  # use 'all' for all depths
    max_length: int = 0  # set to 0 for no truncation cap
    train_batch_size: int = 30
    eval_batch_size: int = 30
    train_max_samples: int = 10000
    val_max_samples: int = 5000
    k_epochs: int = 3
    eval_checks_per_epoch: int = 4  # number of mini-eval checks per epoch
    mini_eval_batches_per_check: int = 2  # batches per mini-eval check
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    seed: int = 42
    logic_dim: int = 256
    num_gates: int = 8
    cross_attn_heads: int = 4
    alpha_init: float = 0.01
    gate_window_axis: str = 'inter_token'  # choose one: 'inter_token' or 'intra_token'
    use_param_matched_baseline: bool = True  # if True, baseline_llama uses no-gate control with matched layer capacity
    param_matched_no_gate_update_type: str = 'mlp'  # one of: 'mlp', 'linear'
    use_bf16_if_available: bool = True
    wandb_project: str = 'ai535-logic-vs-llama'
    wandb_group: str = 'v4-eval-sequential'

CFG = ExpConfig()
if CFG.gate_window_axis not in {'inter_token', 'intra_token'}:
    raise ValueError("CFG.gate_window_axis must be one of {'inter_token', 'intra_token'}")
print(CFG)

In [ ]:
# Imports + reproducibility + device
import gc
import math
import random
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from logic.core.baseline_model import BaselineModel
from logic.core.logic_llama_model import LogicLlamaModel
from logic.core.data_utils import load_proofwriter


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

use_bf16 = bool(CFG.use_bf16_if_available and torch.cuda.is_available() and torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print('AMP enabled:', torch.cuda.is_available(), 'dtype:', amp_dtype)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available() and not use_bf16)

In [ ]:
# ProofWriter data loaders
# max_length=0 means no truncation cap; keep dynamic per-batch padding.
effective_max_length = None if int(CFG.max_length) == 0 else int(CFG.max_length)

train_ds, collate_fn = load_proofwriter(
    model_name=CFG.model_name,
    max_length=effective_max_length,
    depth=CFG.proofwriter_depth,
    split='train',
    config_name=CFG.proofwriter_config,
    max_samples=CFG.train_max_samples,
    seed=CFG.seed,
)
val_ds, _ = load_proofwriter(
    model_name=CFG.model_name,
    max_length=effective_max_length,
    depth=CFG.proofwriter_depth,
    split='validation',
    config_name=CFG.proofwriter_config,
    max_samples=CFG.val_max_samples,
    seed=CFG.seed,
)

train_loader = DataLoader(train_ds, batch_size=CFG.train_batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CFG.eval_batch_size, shuffle=False, collate_fn=collate_fn)

print('Effective max_length:', effective_max_length)
print('Train samples:', len(train_ds), 'Val samples:', len(val_ds))
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

In [ ]:
# Build models (same head shape via num_labels=3)
NUM_LABELS = 3
VALID_MODELS = {'baseline_llama', 'logic_llama'}

def build_baseline() -> torch.nn.Module:
    if CFG.use_param_matched_baseline:
        # Matched baseline path: no-gate stream (no cross-attention), H->L projection per layer, MLP fusion.
        model = LogicLlamaModel(
            model_name=CFG.model_name,
            stream_logic_during_backbone=True,
            logic_dim=CFG.logic_dim,
            num_gates=CFG.num_gates,
            cross_attn_heads=CFG.cross_attn_heads,
            gate_window_axis=CFG.gate_window_axis,
            use_no_gate_stream=True,
            no_gate_update_type=CFG.param_matched_no_gate_update_type,
            no_gate_match_logic_params=True,
            routing_top_k=None,
            routing_train_mode='dense',
            routing_train_cutoff=0.0,
            fusion_mode='mlp',
            alpha_init=CFG.alpha_init,
            num_labels=NUM_LABELS,
        )
        return model.to(device)

    model = BaselineModel(
        model_name=CFG.model_name,
        num_labels=NUM_LABELS,
    )
    return model.to(device)


def build_logic() -> torch.nn.Module:
    model = LogicLlamaModel(
        model_name=CFG.model_name,
        stream_logic_during_backbone=True,
        logic_dim=CFG.logic_dim,
        num_gates=CFG.num_gates,
        cross_attn_heads=CFG.cross_attn_heads,
        gate_window_axis=CFG.gate_window_axis,
        routing_top_k=None,
        routing_train_mode='dense',
        routing_train_cutoff=0.0,
        fusion_mode='linear_bridge',
        alpha_init=CFG.alpha_init,
        num_labels=NUM_LABELS,
    )
    return model.to(device)


def build_model(model_name: str) -> tuple[str, torch.nn.Module]:
    model_name = model_name.strip().lower()
    if model_name not in VALID_MODELS:
        raise ValueError(f"model_name must be one of {sorted(VALID_MODELS)}, got '{model_name}'")
    if model_name == 'baseline_llama':
        return model_name, build_baseline()
    return model_name, build_logic()


selected_model_name = CFG.run_model.strip().lower()
if selected_model_name not in VALID_MODELS:
    raise ValueError(f"CFG.run_model must be one of {sorted(VALID_MODELS)}, got '{CFG.run_model}'")

print('Planned single-model run:', selected_model_name)
print('Gate comparison axis:', CFG.gate_window_axis)
if selected_model_name == 'baseline_llama' and CFG.use_param_matched_baseline:
    print('Baseline mode: parameter-matched no-gate control is ENABLED (cross-attention is DISABLED)')

In [ ]:
# Train/eval helpers
def _safe_float(value, default: float = float('nan')) -> float:
    try:
        x = float(value)
        return x if math.isfinite(x) else default
    except Exception:
        return default


def _compute_grad_norm(model: torch.nn.Module) -> float:
    sq_sum = 0.0
    found = False
    for p in model.parameters():
        if p.grad is None:
            continue
        grad = p.grad.detach()
        sq_sum += float(torch.sum(grad * grad).item())
        found = True
    if not found:
        return 0.0
    return float(math.sqrt(max(sq_sum, 0.0)))


def _count_trainable_params(model: torch.nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def _read_gpu_power_w() -> float | None:
    if not torch.cuda.is_available():
        return None
    # Try NVML first; fallback to nvidia-smi query if NVML bindings are unavailable.
    try:
        import pynvml  # type: ignore
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        power_mw = pynvml.nvmlDeviceGetPowerUsage(handle)
        return float(power_mw) / 1000.0
    except Exception:
        pass
    try:
        import subprocess
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip().splitlines()
        if not out:
            return None
        return float(out[0].strip())
    except Exception:
        return None


def _extract_routing_entropy(outputs) -> float | None:
    history = getattr(outputs, 'routing_history', None)
    if not history:
        return None
    entropies = []
    for weights in history:
        if not isinstance(weights, torch.Tensor) or weights.numel() == 0:
            continue
        probs = weights.detach().float()
        probs = probs.clamp_min(1e-12)
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        entropy = -(probs * probs.log()).sum(dim=-1).mean()
        if torch.isfinite(entropy):
            entropies.append(float(entropy.item()))
    if not entropies:
        return None
    return float(sum(entropies) / len(entropies))


def _extract_fusion_alpha(model: torch.nn.Module) -> float | None:
    fusion = getattr(model, 'fusion', None)
    if fusion is None:
        return None
    alpha = getattr(fusion, 'alpha', None)
    if alpha is None:
        return None
    if isinstance(alpha, torch.Tensor):
        alpha = alpha.detach().float().mean().item()
    return _safe_float(alpha, default=float('nan'))


def _aggregate_checks(values: list[float]) -> dict:
    arr = np.asarray(values, dtype=np.float64)
    return {
        'mean': float(np.mean(arr)),
        'std': float(np.std(arr)),
        'min': float(np.min(arr)),
        'max': float(np.max(arr)),
    }


@torch.no_grad()
def evaluate_model(model: torch.nn.Module, loader: DataLoader, max_batches: int | None = None) -> dict:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    routing_entropy_vals: list[float] = []

    for batch_idx, batch in enumerate(loader):
        if max_batches is not None and batch_idx >= int(max_batches):
            break

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits
            loss = F.cross_entropy(logits, labels)

        total_loss += float(loss.item()) * input_ids.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += int((preds == labels).sum().item())
        total_count += int(input_ids.size(0))

        entropy = _extract_routing_entropy(out)
        if entropy is not None and math.isfinite(entropy):
            routing_entropy_vals.append(float(entropy))

    return {
        'loss': total_loss / max(total_count, 1),
        'acc': total_correct / max(total_count, 1),
        'routing_entropy': (sum(routing_entropy_vals) / len(routing_entropy_vals)) if routing_entropy_vals else None,
    }


def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    trainable_params: int,
    train_flop_factor: float = 6.0,
) -> dict:
    model.train()
    total_loss = 0.0
    total_count = 0
    total_tokens = 0
    total_grad_norm = 0.0
    grad_steps = 0
    non_finite_loss_count = 0
    routing_entropy_vals: list[float] = []
    energy_gpu_wh = 0.0
    power_samples: list[float] = []

    epoch_start = time.perf_counter()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for batch in loader:
        batch_start = time.perf_counter()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits
            loss = F.cross_entropy(logits, labels)

        if not torch.isfinite(loss):
            non_finite_loss_count += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = _compute_grad_norm(model)
        scaler.step(optimizer)
        scaler.update()

        entropy = _extract_routing_entropy(out)
        if entropy is not None and math.isfinite(entropy):
            routing_entropy_vals.append(float(entropy))

        total_grad_norm += float(grad_norm)
        grad_steps += 1
        batch_size = int(input_ids.size(0))
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size
        total_tokens += int(attention_mask.sum().item())

        batch_sec = max(time.perf_counter() - batch_start, 1e-9)
        power_w = _read_gpu_power_w()
        if power_w is not None and math.isfinite(power_w) and power_w >= 0.0:
            power_samples.append(float(power_w))
            energy_gpu_wh += float(power_w) * batch_sec / 3600.0

    epoch_sec = max(time.perf_counter() - epoch_start, 1e-9)
    peak_gpu_mem_gb = (torch.cuda.max_memory_allocated() / (1024 ** 3)) if torch.cuda.is_available() else 0.0
    avg_grad_norm = total_grad_norm / max(grad_steps, 1)
    gpu_avg_power_w = (sum(power_samples) / len(power_samples)) if power_samples else None

    # FLOPs estimate: ~6 * trainable_params * tokens for full training step accounting.
    flops_per_token_est = float(train_flop_factor * trainable_params)
    flops_per_epoch_est = float(flops_per_token_est * total_tokens)
    flops_per_step_est = float(flops_per_epoch_est / max(grad_steps, 1))

    return {
        'loss': total_loss / max(total_count, 1),
        'grad_norm': avg_grad_norm,
        'non_finite_loss_count': int(non_finite_loss_count),
        'samples_per_sec': total_count / epoch_sec,
        'tokens_per_sec': total_tokens / epoch_sec,
        'epoch_sec': epoch_sec,
        'peak_gpu_mem_gb': float(peak_gpu_mem_gb),
        'routing_entropy': (sum(routing_entropy_vals) / len(routing_entropy_vals)) if routing_entropy_vals else None,
        'flops_per_token_est': flops_per_token_est,
        'flops_per_step_est': flops_per_step_est,
        'flops_per_epoch_est': flops_per_epoch_est,
        'energy_gpu_wh': float(energy_gpu_wh),
        'gpu_avg_power_w': gpu_avg_power_w,
    }


def run_training(model: torch.nn.Module, name: str, epochs: int, wandb_run=None) -> list[dict]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    history = []
    train_start = time.perf_counter()
    trainable_params = _count_trainable_params(model)
    cumulative_flops_est = 0.0
    cumulative_energy_gpu_wh = 0.0
    eval_checks = max(1, int(CFG.eval_checks_per_epoch))
    mini_eval_batches = max(1, int(CFG.mini_eval_batches_per_check))

    init_val = evaluate_model(model, val_loader)
    init_row = {
        'epoch': 0,
        'train_loss': None,
        'val_loss': init_val['loss'],
        'val_acc': init_val['acc'],
        'routing_entropy': init_val['routing_entropy'],
        'fusion_alpha': _extract_fusion_alpha(model),
        'flops_per_token_est': float(6.0 * trainable_params),
        'flops_per_step_est': None,
        'flops_per_epoch_est': 0.0,
        'energy_gpu_wh': 0.0,
        'gpu_avg_power_w': None,
        'val_checks_per_epoch': eval_checks,
        'mini_eval_batches_per_check': mini_eval_batches,
        'val_loss_std': 0.0,
        'val_loss_min': init_val['loss'],
        'val_loss_max': init_val['loss'],
        'val_acc_std': 0.0,
        'val_acc_min': init_val['acc'],
        'val_acc_max': init_val['acc'],
        'val_loss_mini_mean': init_val['loss'],
        'val_acc_mini_mean': init_val['acc'],
    }
    history.append(init_row)
    print(f'[{name}] epoch=0 val_loss={init_val["loss"]:.4f} val_acc={init_val["acc"]:.4f}')
    if wandb_run is not None:
        wandb_run.log({
            'epoch': 0,
            'train/loss': float('nan'),
            'val/loss': _safe_float(init_val['loss']),
            'val/acc': _safe_float(init_val['acc']),
            'val/loss_std': 0.0,
            'val/loss_min': _safe_float(init_val['loss']),
            'val/loss_max': _safe_float(init_val['loss']),
            'val/acc_std': 0.0,
            'val/acc_min': _safe_float(init_val['acc']),
            'val/acc_max': _safe_float(init_val['acc']),
            'val/loss_mini_mean': _safe_float(init_val['loss']),
            'val/acc_mini_mean': _safe_float(init_val['acc']),
            'eval/checks_per_epoch': float(eval_checks),
            'eval/mini_eval_batches_per_check': float(mini_eval_batches),
            'train/routing_entropy': _safe_float(init_val['routing_entropy']),
            'train/fusion_alpha': _safe_float(init_row['fusion_alpha']),
            'val/lr': _safe_float(optimizer.param_groups[0]['lr']),
            'reliability/non_finite_loss_count': 0.0,
            'perf/samples_per_sec': float('nan'),
            'perf/tokens_per_sec': float('nan'),
            'time/epoch_sec': 0.0,
            'system/peak_gpu_mem_gb': 0.0,
            'perf/flops_per_token_est': _safe_float(init_row['flops_per_token_est']),
            'perf/flops_per_step_est': float('nan'),
            'perf/flops_per_epoch_est': 0.0,
            'energy/gpu_wh': 0.0,
            'energy/gpu_avg_power_w': float('nan'),
        }, step=0)

    for epoch in range(1, epochs + 1):
        train_stats = train_one_epoch(
            model,
            train_loader,
            optimizer,
            trainable_params=trainable_params,
            train_flop_factor=6.0,
        )

        mini_val_loader = DataLoader(
            val_ds,
            batch_size=CFG.eval_batch_size,
            shuffle=True,
            collate_fn=collate_fn,
        )
        mini_checks_rows = [
            evaluate_model(model, mini_val_loader, max_batches=mini_eval_batches)
            for _ in range(eval_checks)
        ]
        mini_loss_stats = _aggregate_checks([row['loss'] for row in mini_checks_rows])
        mini_acc_stats = _aggregate_checks([row['acc'] for row in mini_checks_rows])

        full_val = evaluate_model(model, val_loader)

        routing_candidates = [row['routing_entropy'] for row in mini_checks_rows if row['routing_entropy'] is not None]
        if full_val['routing_entropy'] is not None:
            routing_candidates.append(full_val['routing_entropy'])
        val_routing_entropy = float(np.mean(routing_candidates)) if routing_candidates else None

        fusion_alpha = _extract_fusion_alpha(model)
        cumulative_flops_est += float(train_stats['flops_per_epoch_est'])
        cumulative_energy_gpu_wh += float(train_stats['energy_gpu_wh'])
        row = {
            'epoch': epoch,
            'train_loss': train_stats['loss'],
            'val_loss': full_val['loss'],
            'val_acc': full_val['acc'],
            'val_loss_std': mini_loss_stats['std'],
            'val_loss_min': mini_loss_stats['min'],
            'val_loss_max': mini_loss_stats['max'],
            'val_acc_std': mini_acc_stats['std'],
            'val_acc_min': mini_acc_stats['min'],
            'val_acc_max': mini_acc_stats['max'],
            'val_loss_mini_mean': mini_loss_stats['mean'],
            'val_acc_mini_mean': mini_acc_stats['mean'],
            'val_checks_per_epoch': eval_checks,
            'mini_eval_batches_per_check': mini_eval_batches,
            'routing_entropy': val_routing_entropy if val_routing_entropy is not None else train_stats['routing_entropy'],
            'fusion_alpha': fusion_alpha,
            'grad_norm': train_stats['grad_norm'],
            'lr': optimizer.param_groups[0]['lr'],
            'non_finite_loss_count': train_stats['non_finite_loss_count'],
            'samples_per_sec': train_stats['samples_per_sec'],
            'tokens_per_sec': train_stats['tokens_per_sec'],
            'epoch_sec': train_stats['epoch_sec'],
            'peak_gpu_mem_gb': train_stats['peak_gpu_mem_gb'],
            'flops_per_token_est': train_stats['flops_per_token_est'],
            'flops_per_step_est': train_stats['flops_per_step_est'],
            'flops_per_epoch_est': train_stats['flops_per_epoch_est'],
            'energy_gpu_wh': train_stats['energy_gpu_wh'],
            'gpu_avg_power_w': train_stats['gpu_avg_power_w'],
        }
        history.append(row)
        print(
            f'[{name}] epoch={epoch} '
            f'train_loss={row["train_loss"]:.4f} '
            f'val_loss(full)={row["val_loss"]:.4f} '
            f'val_acc(full)={row["val_acc"]:.4f} '
            f'mini_acc={row["val_acc_mini_mean"]:.4f}±{row["val_acc_std"]:.4f}'
        )
        if wandb_run is not None:
            wandb_run.log({
                'epoch': epoch,
                'train/loss': _safe_float(row['train_loss']),
                'val/loss': _safe_float(row['val_loss']),
                'val/acc': _safe_float(row['val_acc']),
                'val/loss_std': _safe_float(row['val_loss_std']),
                'val/loss_min': _safe_float(row['val_loss_min']),
                'val/loss_max': _safe_float(row['val_loss_max']),
                'val/acc_std': _safe_float(row['val_acc_std']),
                'val/acc_min': _safe_float(row['val_acc_min']),
                'val/acc_max': _safe_float(row['val_acc_max']),
                'val/loss_mini_mean': _safe_float(row['val_loss_mini_mean']),
                'val/acc_mini_mean': _safe_float(row['val_acc_mini_mean']),
                'eval/checks_per_epoch': _safe_float(row['val_checks_per_epoch']),
                'eval/mini_eval_batches_per_check': _safe_float(row['mini_eval_batches_per_check']),
                'train/grad_norm': _safe_float(row['grad_norm']),
                'val/lr': _safe_float(row['lr']),
                'reliability/non_finite_loss_count': _safe_float(row['non_finite_loss_count']),
                'perf/samples_per_sec': _safe_float(row['samples_per_sec']),
                'perf/tokens_per_sec': _safe_float(row['tokens_per_sec']),
                'time/epoch_sec': _safe_float(row['epoch_sec']),
                'system/peak_gpu_mem_gb': _safe_float(row['peak_gpu_mem_gb']),
                'train/routing_entropy': _safe_float(row['routing_entropy']),
                'train/fusion_alpha': _safe_float(row['fusion_alpha']),
                'perf/flops_per_token_est': _safe_float(row['flops_per_token_est']),
                'perf/flops_per_step_est': _safe_float(row['flops_per_step_est']),
                'perf/flops_per_epoch_est': _safe_float(row['flops_per_epoch_est']),
                'energy/gpu_wh': _safe_float(row['energy_gpu_wh']),
                'energy/gpu_avg_power_w': _safe_float(row['gpu_avg_power_w']),
            }, step=epoch)

    total_train_sec = max(time.perf_counter() - train_start, 0.0)
    if wandb_run is not None:
        wandb_run.summary['time/total_train_sec'] = float(total_train_sec)
        wandb_run.summary['perf/flops_total_est'] = float(cumulative_flops_est)
        wandb_run.summary['perf/trainable_params'] = float(trainable_params)
        wandb_run.summary['energy/gpu_wh_total'] = float(cumulative_energy_gpu_wh)
        wandb_run.summary['eval/checks_per_epoch'] = float(eval_checks)
        wandb_run.summary['eval/mini_eval_batches_per_check'] = float(mini_eval_batches)
        if history:
            wandb_run.summary['summary.val/acc'] = _safe_float(history[-1]['val_acc'])

    return history

## Train

In [ ]:
# Run one selected model across multiple seeds for K epochs (sequentially)
K_EPOCHS = CFG.k_epochs
base_model_name = selected_model_name
seeds_to_run = [int(s) for s in CFG.run_seeds] if CFG.run_seeds else [int(CFG.seed)]
seeds_to_run = list(dict.fromkeys(seeds_to_run))  # keep order, remove duplicates

print('Running model:', base_model_name)
print('Gate comparison axis:', CFG.gate_window_axis)
print('Seeds:', seeds_to_run)

history = []
param_count = None
run_name = base_model_name

for run_seed in seeds_to_run:
    print(f'\n=== Seed {run_seed} ===')
    set_seed(run_seed)

    run_name, model = build_model(base_model_name)
    baseline_is_param_matched = bool(
        run_name == 'baseline_llama' and CFG.use_param_matched_baseline
    )
    no_gate_uses_cross_attn = False if baseline_is_param_matched else None
    if param_count is None:
        param_count = int(sum(p.numel() for p in model.parameters()))
        print('Params:', param_count)

    run_display_name = f'{run_name}-seed{run_seed}'
    run = wandb.init(
        project=CFG.wandb_project,
        group=CFG.wandb_group,
        name=run_display_name,
        reinit=True,
        config={
            'wandb': {
                'group': CFG.wandb_group,
                'run_name': run_display_name,
            },
            'model': {
                'name': CFG.model_name,
                'variant': run_name,
                'variant_note': 'baseline_param_matched' if baseline_is_param_matched else 'standard',
                'gate_mode': 'soft',
                'gate_window_axis': CFG.gate_window_axis,
                'routing_top_k': None,
                'use_no_gate_stream': baseline_is_param_matched,
                'use_param_matched_baseline': baseline_is_param_matched,
                'param_matched_no_gate_update_type': CFG.param_matched_no_gate_update_type if baseline_is_param_matched else None,
                'no_gate_uses_cross_attn': no_gate_uses_cross_attn,
                'fusion_mode': 'mlp' if baseline_is_param_matched else 'linear_bridge',
                'lora': {'enabled': False},
                'logic_dim': CFG.logic_dim,
                'num_gates': CFG.num_gates,
                'cross_attn_heads': CFG.cross_attn_heads,
                'alpha_init': CFG.alpha_init,
                'capacity_window_axis': CFG.gate_window_axis,
            },
            'train': {
                'epochs': CFG.k_epochs,
                'batch_size': CFG.train_batch_size,
                'eval_batch_size': CFG.eval_batch_size,
                'learning_rate': CFG.learning_rate,
                'weight_decay': CFG.weight_decay,
                'seed': run_seed,
                'max_length': CFG.max_length,
                'use_bf16_if_available': CFG.use_bf16_if_available,
            },
            'data': {
                'proofwriter_config': CFG.proofwriter_config,
                'proofwriter_depth': CFG.proofwriter_depth,
                'train_max_samples': CFG.train_max_samples,
                'val_max_samples': CFG.val_max_samples,
            },
        },
    )

    try:
        seed_history = run_training(model, name=run_name, epochs=K_EPOCHS, wandb_run=run)
        for row in seed_history:
            row['seed'] = run_seed
            row['model'] = run_name
        history.extend(seed_history)
    finally:
        run.finish()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print('\nCompleted seeds:', seeds_to_run)

In [ ]:
# Show multi-seed results + plot val_acc / num_parameters
import matplotlib.pyplot as plt
import pandas as pd

results = pd.DataFrame(history).copy()
if 'model' not in results.columns:
    results['model'] = run_name
if 'seed' not in results.columns:
    results['seed'] = int(CFG.seed)

results['num_parameters'] = int(param_count)
results['val_acc_per_param'] = results['val_acc'] / results['num_parameters']

plot_df = results[results['epoch'] > 0].dropna(subset=['val_acc_per_param']).copy()
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))

    for seed_value, g in plot_df.groupby('seed', sort=True):
        ax.plot(
            g['epoch'],
            g['val_acc_per_param'],
            marker='o',
            alpha=0.35,
            label=f'seed {seed_value}',
        )

    mean_curve = plot_df.groupby('epoch', as_index=False)['val_acc_per_param'].mean()
    ax.plot(
        mean_curve['epoch'],
        mean_curve['val_acc_per_param'],
        color='black',
        linewidth=2.5,
        marker='o',
        label='seed mean',
    )

    ax.set_title('Validation Accuracy per Parameter (Per Seed + Mean)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('val_acc / num_parameters')
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

results = results[[
    'model',
    'seed',
    'epoch',
    'num_parameters',
    'val_acc_per_param',
    'train_loss',
    'val_loss',
    'val_acc',
    'val_loss_mini_mean',
    'val_acc_mini_mean',
    'val_loss_std',
    'val_loss_min',
    'val_loss_max',
    'val_acc_std',
    'val_acc_min',
    'val_acc_max',
    'val_checks_per_epoch',
    'mini_eval_batches_per_check',
    'routing_entropy',
    'fusion_alpha',
    'grad_norm',
    'lr',
    'non_finite_loss_count',
    'samples_per_sec',
    'tokens_per_sec',
    'epoch_sec',
    'peak_gpu_mem_gb',
    'energy_gpu_wh',
    'gpu_avg_power_w',
    'flops_per_token_est',
    'flops_per_step_est',
    'flops_per_epoch_est',
]]
results.sort_values(['seed', 'epoch']).reset_index(drop=True)

## Mechanism Visuals (Routing + Gate Families)

These plots mirror the quick introspection style from logic_hf.ipynb, but run directly in this V4 workflow.

What this section shows for one validation batch:
1. Layer-by-channel routing heatmap
2. AND / OR / NOT family channel means (if routing channels are grouped as 3G)
3. Cross-attention entropy per layer (logic model path)

In [ ]:
# Mechanism visuals on one batch (safe for both logic and baseline selections)
import matplotlib.pyplot as plt
import numpy as np

viz_name, viz_model = build_model(selected_model_name)
viz_model.eval()

viz_batch = next(iter(val_loader))
viz_input_ids = viz_batch['input_ids'].to(device)
viz_attention_mask = viz_batch['attention_mask'].to(device)

with torch.no_grad():
    viz_out = viz_model(input_ids=viz_input_ids, attention_mask=viz_attention_mask)

routing_history = getattr(viz_out, 'routing_history', None)
cross_attn_history = getattr(viz_out, 'cross_attn_history', None)

if not routing_history:
    print('No routing history available for this selected model; skipping mechanism visuals.')
else:
    # [num_layers, channels] by averaging over tokens for first batch item
    route_mat = torch.stack([rw[0].detach().float().mean(dim=0) for rw in routing_history]).cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(route_mat, aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_title(f'Routing Weights Heatmap ({viz_name})')
    ax.set_xlabel('Routing channel index')
    ax.set_ylabel('Transformer layer')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    # If channels are grouped as AND/OR/NOT banks (3G), show per-family channel bars.
    if route_mat.shape[1] >= 3 and route_mat.shape[1] % 3 == 0:
        g = route_mat.shape[1] // 3
        and_mean = route_mat[:, :g].mean(axis=0)
        or_mean = route_mat[:, g:2 * g].mean(axis=0)
        not_mean = route_mat[:, 2 * g:].mean(axis=0)

        fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), sharey=True)
        for ax, vals, title, color in [
            (axes[0], and_mean, 'AND family mean routing', '#2f6db3'),
            (axes[1], or_mean, 'OR family mean routing', '#3ca370'),
            (axes[2], not_mean, 'NOT family mean routing', '#c44e52'),
        ]:
            ax.bar(np.arange(len(vals)), vals, color=color, edgecolor='black', linewidth=0.4)
            ax.set_title(title)
            ax.set_xlabel('Channel index within family')
            ax.grid(axis='y', alpha=0.25)
        axes[0].set_ylabel('Mean routing weight')
        plt.tight_layout()
        plt.show()
    else:
        print('Routing channels are not in a 3-family layout; skipped AND/OR/NOT family bars.')

    if cross_attn_history:
        # Normalized entropy trend per layer for quick attention sharpness diagnostics.
        layer_entropy = []
        for probs in cross_attn_history:
            p = probs[0].detach().float().clamp_min(1e-12)
            p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            ent = -(p * p.log()).sum(dim=-1).mean().item()
            layer_entropy.append(ent)

        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(np.arange(len(layer_entropy)), layer_entropy, marker='o', color='black')
        ax.set_title('Cross-Attention Entropy by Layer')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Entropy')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No cross-attention history for this run path (expected for no-gate baseline mode).')

del viz_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## What To Tune Next

If runtime is too slow on Colab:
1. reduce `train_max_samples` and `val_max_samples`
2. reduce `k_epochs`
3. lower `max_length`
4. keep batch size at 1

If memory is still too high for full unfrozen Llama-3.1-8B, move to a larger GPU runtime or temporarily switch to a smaller real checkpoint to validate workflow.